In [ ]:
import importlib, Extraction as EX
import pandas as pd
importlib.reload(EX)
DATASETS, CSV_HASHES = EX.discover_processed_datasets(show_info=True)
print(sorted(DATASETS), {k: len(v) for k, v in DATASETS.items()})

In [ ]:
DATASETS, CSV_HASHES = EX.discover_processed_datasets(
    datasets_root=EX.PROCESSED_DATASETS_ROOT,
    show_info=True,
)

In [ ]:
for name, df in DATASETS.items():
    assert list(df.columns) == list(EX.PROCESSED_COLUMNS)
    assert df.index.is_unique and df.index[0] == 0 and df.index[-1] == len(df) - 1
    assert df["label"].map(lambda x: isinstance(x, list) and len(x) > 0).all()

In [ ]:
EX.configure_external_storage(show_info=True)
VALIDITY_DATASETS = {name: df.iloc[:1].copy() for name, df in DATASETS.items()}
results = EX.run_model_matrix(
    datasets=VALIDITY_DATASETS,
    dataset_csv_hashes=None,
    groups=None,
    experiment_id="tests_v1",
    pooling="mean",
    max_length=32,
    use_half_precision=False,
    flush_every_batches=1,
    continue_on_model_error=True,
    show_verbose=False,
    show_info=True,
    show_critical=True,
    show_debug=False,
)

NEW_EXPERIMENT_ID = "master_v1"
results = EX.run_model_matrix(
    datasets=DATASETS,
    dataset_csv_hashes=CSV_HASHES,
    groups=None,
    experiment_id=NEW_EXPERIMENT_ID,
    pooling="mean",
    max_length=512,
    use_half_precision=True,
    flush_every_batches=8,
    continue_on_model_error=True,
    show_verbose=True,
    show_info=True,
    show_critical=True,
    show_debug=True,
)

In [ ]:
from sys import audit
df_audit = pd.DataFrame(audit)

In [ ]:
df_audit = pd.DataFrame(EX.audit_experiment(datasets=DATASETS, show_details=False))

In [ ]:
assert len(DATASETS["amazon_polarity"]) <= 200_000, (
    "amazon_polarity is too large; subsample first (see notes)"
)

# 1. Force the chooser on the problematic dataset
python3 master_dataset.py process hf://shreyaspullehf/emotion_dataset_100k \
    --choose-columns \
    -o /Volumes/Amirali/datasets/emotion_100k/processed/emotion_100k_clean.csv \
    --overwrite

# 2. Confirm the sidecar recorded the choice
cat /Volumes/Amirali/datasets/emotion_dataset_100k/schema.json
# expected: "text_column": "col_0", "label_column": "col_2"

# 3. Re-run without --choose-columns; it should skip the prompt
python3 master_dataset.py process hf://shreyaspullehf/emotion_dataset_100k \
    -o /Volumes/Amirali/datasets/emotion_100k/processed/emotion_100k_clean.csv \
    --overwrite
# expected: no prompts, detection reads the sidecar

# 4. Force a re-pick by deleting the sidecar
rm /Volumes/Amirali/datasets/emotion_dataset_100k/schema.json
python3 master_dataset.py process hf://shreyaspullehf/emotion_dataset_100k \
    --choose-columns \
    -o /Volumes/Amirali/datasets/emotion_100k/processed/emotion_100k_clean.csv \
    --overwrite
# expected: prompts again